# GSE138458 — Does the Autoencoder's Latent Space Capture Disease Structure?

The project's core question: **what do learned representations of gene expression capture that
simpler, interpretable representations do not?** Day 1 found that raw-feature PCA does *not*
separate SLE from control — the dominant unsupervised variance tracks interferon-signature
heterogeneity, not diagnosis. Day 2's classical baselines separate the classes very well when
*supervised*, but SHAP's gene attributions are diffuse and share no genes with Day 1's
top-variance list. Day 3 trained a 32-dimensional autoencoder, purely unsupervised (no label in
the loss), on the same 2000 `SelectKBest`-selected, scaled genes the classical baselines use.

This notebook asks the question those three days set up: **does that 32-dim bottleneck separate
SLE from control better than raw-feature PCA, and do the genes driving any such separation
overlap with SHAP's top features — or are they similarly diffuse?**

Reusable logic lives in `src/`: `biomedical_ml.ae_training` (checkpoint loading, latent
extraction), `biomedical_ml.eda` (PCA, gene-correlation ranking — reused unchanged from Day 1),
and `biomedical_ml.evaluation` (the repeated grouped-CV machinery, extended in Day 4 to accept
any pipeline, not just the registered classical baselines). This notebook is visualization,
comparison, and narrative.

In [ ]:
from __future__ import annotations

import json
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import umap
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from biomedical_ml.ae_training import (
    build_model_from_checkpoint,
    encode_all,
    load_checkpoint,
    transform_with_checkpoint,
)
from biomedical_ml.config import FIGURES_DIR, RESULTS_DIR, SEED, ensure_dirs, load_config, set_seed
from biomedical_ml.eda import correlate_features_with_target, pca_embedding
from biomedical_ml.evaluation import _best_threshold, evaluate_pipeline_factory, summarize_results
from biomedical_ml.preprocessing import build_dataset
from biomedical_ml.splits import holdout_split

set_seed()
ensure_dirs()
sns.set_theme(style="whitegrid", font_scale=0.9)
%matplotlib inline

# Benign: UMAP warns about n_jobs when random_state is set (it disables its own
# parallelism for reproducibility, which is exactly what we want).
warnings.filterwarnings("ignore", message=".*n_jobs value.*")

cfg = load_config()
N_SPLITS = cfg["split"]["n_splits"]
N_REPEATS = cfg["split"]["n_repeats"]

CASE_COLOUR = "#c44e52"
CONTROL_COLOUR = "#4c72b0"
PALETTE = {"SLE": CASE_COLOUR, "Control": CONTROL_COLOUR}

## Load the dataset, the trained autoencoder, and Day 1-2's results

In [ ]:
dataset = build_dataset(annotated_only=True)
checkpoint = load_checkpoint(RESULTS_DIR / "ae_checkpoint.pt")
model = build_model_from_checkpoint(checkpoint)

print(dataset.summary())
print(
    f"\nAE checkpoint: latent_dim={checkpoint['autoencoder_config']['latent_dim']}, "
    f"best_epoch={checkpoint['best_epoch'] + 1}, best_val_loss={checkpoint['best_val_loss']:.4f}"
)

day1_summary = json.loads((RESULTS_DIR / "eda_summary.json").read_text(encoding="utf-8"))
day2_summary = json.loads((RESULTS_DIR / "baseline_summary.json").read_text(encoding="utf-8"))
day2_cv_results = pd.read_csv(RESULTS_DIR / "baseline_cv_results.csv")

## Extract latent embeddings for all 330 samples

`encode_all` applies the checkpoint's exact feature selection and standardization, then runs the
frozen encoder — including on the 66 samples the autoencoder only ever saw as validation data,
and even the 264 training samples it was fit on. Neither distinction matters here: the encoder is
now a fixed function, and we're asking a purely descriptive question about the representation it
produces, not re-evaluating the autoencoder's own training.

In [ ]:
latents = encode_all(model, dataset.X, checkpoint)
print(f"latents: {latents.shape[0]} samples x {latents.shape[1]} dimensions")
assert not latents.isna().to_numpy().any()

## Visualizing separation: raw genes vs. selected genes vs. the AE's compression

One confound has to be controlled before comparing "raw-feature PCA" against "latent PCA": the
autoencoder's *input* is already the 2000 genes `SelectKBest` chose using the label — Day 1's raw
PCA used all 31,266 genes with no such filtering. So a fair comparison needs three views, not
two:

- **(a)** all 31,266 genes, centred only — exactly Day 1's PCA.
- **(b)** the same 2000 selected, standardized genes the autoencoder receives, but *not yet
  compressed* — isolates what feature selection alone buys.
- **(c)** the AE's 32-dimensional latent space — isolates what the *compression* adds on top of
  (b).

Plus **(d)**, a UMAP embedding of the latent space, which can reveal non-linear neighbourhood
structure a linear 2D PCA slice of the same space might not show.

In [ ]:
selected_standardized = pd.DataFrame(
    transform_with_checkpoint(dataset.X, checkpoint),
    index=dataset.X.index,
    columns=checkpoint["selected_probe_ids"],
)

raw_scores, raw_var = pca_embedding(dataset.X, n_components=5)
selected_scores, selected_var = pca_embedding(selected_standardized, n_components=5)
latent_scores, latent_var = pca_embedding(latents, n_components=5)

umap_embedding = umap.UMAP(n_components=2, random_state=SEED).fit_transform(latents.to_numpy())
latent_umap = pd.DataFrame(umap_embedding, index=latents.index, columns=["UMAP1", "UMAP2"])

label = dataset.y.map({0: "Control", 1: "SLE"})
panels = [
    (raw_scores, "PC1", "PC2", f"(a) all 31266 genes\nPC1 {raw_var[0]:.1%}, PC2 {raw_var[1]:.1%}"),
    (
        selected_scores, "PC1", "PC2",
        f"(b) 2000 selected genes\n(uncompressed) PC1 {selected_var[0]:.1%}, PC2 {selected_var[1]:.1%}",
    ),
    (
        latent_scores, "PC1", "PC2",
        f"(c) AE latent PCA\nPC1 {latent_var[0]:.1%}, PC2 {latent_var[1]:.1%}",
    ),
    (latent_umap, "UMAP1", "UMAP2", "(d) AE latent UMAP"),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))
for ax, (scores, xcol, ycol, title) in zip(axes, panels, strict=True):
    for name, colour in PALETTE.items():
        mask = (label == name).to_numpy()
        ax.scatter(
            scores.loc[mask, xcol], scores.loc[mask, ycol], s=18, alpha=0.75, c=colour,
            label=f"{name} (n={int(mask.sum())})", edgecolor="white", linewidth=0.3,
        )
    ax.set_title(title, fontsize=10)
    ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "latent_space_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

**(a) shows no separation** — this is Day 1's finding, reproduced exactly. **(b) and (c) look
qualitatively similar**: in both, most control points drift toward one edge of the cloud, but
several still sit inside the SLE cluster — feature selection alone already buys a fair amount of
visual separation, so the 2D PCA slices of (b) and (c) don't obviously distinguish "selection" from
"compression." **(d) is the most visually dramatic**: nearly all control samples form a distinct
cluster, separate from the main SLE cloud. UMAP is a non-linear, neighbourhood-preserving method,
so it can surface structure a linear 2-component PCA slice compresses away — worth treating as a
genuine signal alongside the quantitative checks below, not a substitute for them (UMAP layouts
are also known to visually oversell cluster separation at the wrong hyperparameters, so the next
section checks this with metrics that don't depend on picking two components to look at).

## Quantifying it: silhouette score, computed on the full (not 2D) representation

Silhouette score treats SLE/control as if they were cluster labels and asks how well-separated
they are, using every dimension of a representation — not just whichever two components happen to
get plotted. Computing it on all three of (a), (b), and (c) isolates the same two questions the
visualization raised: does selection help, and does compression help *further*?

In [ ]:
sil_raw = silhouette_score(dataset.X.to_numpy(), dataset.y)
sil_selected = silhouette_score(selected_standardized.to_numpy(), dataset.y)
sil_latent = silhouette_score(latents.to_numpy(), dataset.y)

print(f"(a) all 31266 genes                      : {sil_raw:+.4f}")
print(f"(b) 2000 selected genes, uncompressed     : {sil_selected:+.4f}")
print(f"(c) AE 32-dim latent (compressed from b)  : {sil_latent:+.4f}")

The full-dimensional numbers resolve what the 2D plots left ambiguous: **(a) -0.011 -> (b) 0.196
-> (c) 0.317**. Feature selection alone takes silhouette from essentially zero to a real positive
value (expected — those 2000 genes were chosen because they differ between groups). But the AE's
compression adds a further, comparably-sized jump *on top of that*, using no label information in
its own training. That is the more defensible version of "the latent space separates the classes
better": not latent-vs-raw, but latent-vs-the-same-genes-uncompressed.

One honest caveat: silhouette scores aren't strictly comparable across representations of
different dimensionality (2000-dim vs. 32-dim) — distance concentration in high dimensions can by
itself depress silhouette in (b) relative to (c). This is why the comparison leans on three
converging lines of evidence (visualization, silhouette, and the probe AUC below) rather than any
one number alone.

## A linear probe on the frozen latent space vs. Day 2's classical baselines

Same repeated (5x5), subject-grouped, stratified CV as Day 2 — `evaluate_pipeline_factory` is the
Day 4 extension to `biomedical_ml.evaluation` that makes this a fair, apples-to-apples comparison:
identical splits, identical threshold tuning, identical metrics, just a different (and much
smaller) feature space. The probe itself is deliberately minimal — a scaler and an L2 logistic
regression, nothing SHAP-worthy on its own — because the question is what the *frozen embedding*
carries, not how good a classifier we can build.

In [ ]:
def probe_factory(seed: int) -> Pipeline:
    return Pipeline(
        [
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed)),
        ]
    )


probe_results = evaluate_pipeline_factory(
    "ae_latent_probe", probe_factory, latents, dataset.y, dataset.groups,
    n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=SEED,
)

combined_results = pd.concat([day2_cv_results, probe_results], ignore_index=True)
combined_summary = summarize_results(combined_results)
combined_summary.round(4)

The AE-latent probe lands at **ROC-AUC 0.961 (+/- 0.037)** — squarely inside Day 2's range
(0.943-0.970) and statistically indistinguishable from it, exactly like every pairwise comparison
among Day 2's own five baselines. The genuinely notable part is what it takes to get there: **32
dimensions instead of 2000**, and the **lowest fold-to-fold std of any model tested so far**
(0.037, versus 0.06-0.09 for the classical baselines). A representation that compresses 2000
correlated genes down to 32 numbers, with zero label supervision, and still supports a simple
linear classifier performing this consistently, is doing real work — even though it isn't winning
outright, which continues this project's honest pattern (Day 2 already showed classical models
cluster tightly here) rather than breaking it.

### Robustness check: the probe's CV doesn't respect the AE's own train/val split

`evaluate_pipeline_factory` re-splits all 330 samples fresh on every CV repeat — it has no
awareness of which 264 samples the autoencoder itself trained on. That's a real asymmetry against
Day 2: Day 2's classical pipelines refit `SelectKBest` from scratch every fold, so they never
touch a test fold's raw values at all, in any way. The AE's encoder, by contrast, was fit *once* on
the raw expression of its 264 training subjects before any of the probe's CV folds were drawn — so
some of the probe's "test" folds land on samples the encoder already fit its reconstruction loss
to (never their label, but their raw values). This isn't label leakage, but it means the pooled
0.961 may be a mild overestimate of how the representation generalizes to genuinely novel samples.

The check below trains the probe only on the AE's 264 training subjects and evaluates it only on
the 66 the AE never trained on — the cleanest inductive test available from this one checkpoint.

In [ ]:
train_subjects = set(checkpoint["train_subject_ids"])
val_subjects = set(checkpoint["val_subject_ids"])
train_mask = dataset.groups.isin(train_subjects).to_numpy()
val_mask = dataset.groups.isin(val_subjects).to_numpy()

n_val_control_samples = int((dataset.y[val_mask] == 0).sum())
n_val_control_subjects = dataset.groups[val_mask][dataset.y[val_mask] == 0].nunique()
print(
    f"AE-val-only samples: {val_mask.sum()} "
    f"({int(dataset.y[val_mask].sum())} SLE, {n_val_control_samples} control "
    f"from {n_val_control_subjects} control subjects)"
)

restricted_probe = probe_factory(SEED)
restricted_probe.fit(latents[train_mask], dataset.y[train_mask])

proba_val = restricted_probe.predict_proba(latents[val_mask])[:, 1]
auc_restricted = roc_auc_score(dataset.y[val_mask], proba_val)

proba_train = restricted_probe.predict_proba(latents[train_mask])[:, 1]
threshold = _best_threshold(dataset.y[train_mask].to_numpy(), proba_train)
balanced_acc_restricted = balanced_accuracy_score(dataset.y[val_mask], proba_val >= threshold)

print(
    f"\nProbe trained on AE-train only, evaluated on AE-val-only: "
    f"ROC-AUC = {auc_restricted:.4f}, balanced accuracy = {balanced_acc_restricted:.4f}"
)
print("(pooled repeated-CV headline number: ROC-AUC = 0.9612 +/- 0.0371)")

**Restricted to genuinely-unseen samples: ROC-AUC = 0.936, balanced accuracy = 0.726** — both
lower than the pooled 0.961 / 0.874. The asymmetry above is real and this is its measurable effect.

But this restricted check is not more authoritative than the pooled number — it's built on **only
4 control samples from 3 control subjects**, thinner than anything else in this project (Day 1's
already-thin 22 control subjects, cut down further by an 80/20 split). A single-point AUC on 4
control samples is about as high-variance an estimate as this dataset can produce. The true
generalization gap likely sits somewhere between "zero" and this 0.025 AUC / 0.148
balanced-accuracy drop, and this one checkpoint can't pin it down more precisely than that.

**0.961 remains the headline number** reported above and in the README — this is a documented
robustness caveat on it, not a replacement for it.

## Which genes track the AE's disease-relevant direction?

The encoder is a non-linear MLP, so there's no PCA-style loading vector to read gene importance
off directly. The approach here: fit one probe on a held-out training split (not the repeated-CV
one above — this is a single interpretability fit, exactly mirroring how Day 2 computed SHAP on a
single holdout-trained model rather than per-fold), rank the 32 latent dimensions by the
probe's `|coefficient|`, then correlate every one of the 2000 selected genes against the
single most disease-associated dimension, across all 330 samples.

In [ ]:
train_idx, test_idx = holdout_split(dataset.y, dataset.groups, n_splits=N_SPLITS, seed=SEED)
holdout_probe = probe_factory(SEED)
holdout_probe.fit(latents.iloc[train_idx], dataset.y.iloc[train_idx])

coefs = holdout_probe.named_steps["clf"].coef_[0]
dim_ranking = pd.DataFrame({"dim": latents.columns, "coef": coefs})
dim_ranking["abs_coef"] = dim_ranking["coef"].abs()
dim_ranking = dim_ranking.sort_values("abs_coef", ascending=False).reset_index(drop=True)
print("Top 10 latent dimensions by |probe coefficient|:")
print(dim_ranking.head(10).to_string(index=False))

top_dim = dim_ranking.iloc[0]["dim"]
print(f"\nMost disease-associated dimension: {top_dim}")

gene_ranking = correlate_features_with_target(
    selected_standardized, latents[top_dim], dataset.annotation, n=20
)
gene_ranking[["probe_id", "gene_symbol", "correlation"]]

The top genes correlated with `z2` are dominated by ribosomal proteins and translation machinery —
**RPL18A** (twice, on two different probes), **RPL11**, **RPL41**, **RPS27**, **EEF1B2**,
**TOMM7**, **OAZ1** — all negatively correlated (-0.52 to -0.42), a notably tight range compared to
Day 2's SHAP ranking (which spanned 0.16 down to 0.07, more than 2x). A coherent functional theme
and a tighter correlation spread both suggest this direction is tracking something more unified
than Day 2's diffuse, collinearity-driven SHAP attribution.

One honest, necessary caveat: ribosomal/translation gene modules in whole blood are a classic
signature of shifting immune cell-type proportions (e.g., lymphocyte vs. neutrophil ratios), not
necessarily SLE-specific biology. This result says the AE organizes its most disease-predictive
axis around a coherent gene module — it does not by itself establish what that module means
biologically.

Does this list overlap with Day 1's top-variance genes (dominated by the interferon signature) or
Day 2's top-SHAP genes?

In [ ]:
latent_genes = set(gene_ranking["gene_symbol"].dropna())
variance_genes = set(day1_summary["top_variable_genes"])
shap_genes = {g["gene_symbol"] for g in day2_summary["top_shap_genes"] if g["gene_symbol"]}

overlap_with_variance = latent_genes & variance_genes
overlap_with_shap = latent_genes & shap_genes

print(f"Latent-direction genes ({len(latent_genes)}): {sorted(latent_genes)}")
print(f"\nOverlap with Day 1 top-variance genes: {sorted(overlap_with_variance)} ({len(overlap_with_variance)})")
print(f"Overlap with Day 2 top-SHAP genes:     {sorted(overlap_with_shap)} ({len(overlap_with_shap)})")

**Zero overlap, both ways.** Three days, three different lenses on "what matters" — unsupervised
variance (Day 1: interferon/HLA/globin genes), a regularized linear classifier's SHAP attributions
(Day 2: a diffuse set with no shared function), and an unsupervised autoencoder's most
disease-predictive direction (Day 4: a coherent ribosomal/translation module) — and none of the
three gene lists share a single gene. That is consistent with, and extends, Day 2's collinearity
argument: with this much redundancy across co-expressed genes, there are many different,
non-overlapping subsets of features that each carry enough signal to separate SLE from control.
*Which* subset a given method surfaces depends on that method's own inductive bias (L2's tendency
to spread weight thinly vs. an autoencoder's bottleneck concentrating variance) — not, on this
evidence, on which genes are "truly" driving the disease.

## Summary — answering the README's Day 4 question

**Does the latent space separate SLE from control better than raw-feature PCA?** Yes, clearly —
but the fair comparison isn't latent-vs-raw, it's latent-vs-the-same-genes-uncompressed, since the
autoencoder's input was already `SelectKBest`-filtered using the label. Controlling for that:
silhouette goes from -0.011 (all 31266 genes) to 0.196 (2000 selected genes alone) to 0.317 (after
AE compression) — feature selection and compression each contribute, and compression's
contribution happens with zero label supervision in its own training. UMAP makes this visually
dramatic; a linear probe on the 32-dim space (ROC-AUC 0.961, the *lowest* variance of any model
tested across this project) confirms it quantitatively, at 1.6% of the raw feature count.

**Do the genes driving that separation overlap with SHAP, or are they similarly diffuse?**
Neither, precisely — they're *not diffuse* (a coherent ribosomal/translation module, tightly
correlated) but they *don't overlap* with SHAP's list, or with Day 1's top-variance genes, at all.
Three methods, three non-overlapping answers to "what matters," all separating the classes well.
On data this collinear, that is itself the finding: high predictive performance is compatible with
genuine ambiguity about which specific genes are responsible, regardless of which modelling
approach — classical or representation-learning — is used to find it.

This is the honest result the project brief anticipated: not "the autoencoder wins" or "the
autoencoder loses," but a representation that matches classical performance through a materially
different, more compact, and differently-structured route — which is precisely the kind of
comparison this project set out to make.

The cell below saves the numbers behind this notebook for later reference.

In [ ]:
summary = {
    "silhouette": {
        "all_31266_genes": float(sil_raw),
        "2000_selected_genes_uncompressed": float(sil_selected),
        "ae_latent_32dim": float(sil_latent),
    },
    "pca_variance_ratio": {
        "all_31266_genes": [round(float(v), 4) for v in raw_var[:5]],
        "2000_selected_genes_uncompressed": [round(float(v), 4) for v in selected_var[:5]],
        "ae_latent_32dim": [round(float(v), 4) for v in latent_var[:5]],
    },
    "probe_vs_classical_baselines": {
        model: {col: round(float(val), 4) for col, val in row.items()}
        for model, row in combined_summary.to_dict(orient="index").items()
    },
    "probe_robustness_check": {
        "note": (
            "The pooled CV above re-splits all 330 samples per fold, without regard to which "
            "264 the AE itself trained on. This restricts training/evaluation to the AE's own "
            "train/val split for comparison; see the markdown cells above for the caveats on "
            "reading this number as more than a robustness check."
        ),
        "n_val_samples": int(val_mask.sum()),
        "n_val_control_samples": n_val_control_samples,
        "n_val_control_subjects": int(n_val_control_subjects),
        "roc_auc_restricted_to_ae_val_only": round(float(auc_restricted), 4),
        "balanced_accuracy_restricted_to_ae_val_only": round(float(balanced_acc_restricted), 4),
        "pooled_cv_roc_auc_mean": round(float(combined_summary.loc["ae_latent_probe", "roc_auc_mean"]), 4),
    },
    "top_disease_associated_latent_dim": top_dim,
    "top_latent_dim_coefficients": dim_ranking.head(10).to_dict(orient="records"),
    "genes_correlated_with_top_latent_dim": gene_ranking.to_dict(orient="records"),
    "overlap_with_day1_top_variance_genes": sorted(overlap_with_variance),
    "overlap_with_day2_top_shap_genes": sorted(overlap_with_shap),
}

summary_path = RESULTS_DIR / "representation_comparison_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"wrote {summary_path}")